In [ ]:
# =============================================================================
# NOTEBOOK 3: UNSUPERVISED MODELLING
# =============================================================================
# Purpose: Apply DBSCAN and HDBSCAN clustering algorithms to discover
# traffic pattern clusters. Use PCA for dimensionality reduction.
# Evaluate clustering quality using internal metrics.
# =============================================================================


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
import hdbscan
import json
import pickle
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:


# Load engineered features
X_scaled = pd.read_hdf('../data/features_scaled.h5')
temporal_df = pd.read_hdf('../data/temporal_features.h5')

with open('../data/feature_names.json', 'r') as f:
    feature_names = json.load(f)

print(f"Feature matrix loaded: {X_scaled.shape}")
print(f"Temporal features loaded: {temporal_df.shape}")
print(f"Number of features: {len(feature_names)}")


# ---- Shared Helper: Safe Silhouette Score ----
# Passing sample_size to sklearn's silhouette_score lets it draw its own
# internal subsample AFTER our guard has run — that subsample can still
# collapse to a single cluster by bad luck and re-raise the ValueError.
# Fix: subsample manually so we can re-check label diversity on the exact
# array being passed, then call silhouette_score with no sample_size arg.

def safe_silhouette(X, labels, sample_size=5000, random_state=42):
    """
    Compute silhouette score ignoring noise points (label == -1).
    Subsamples manually so the diversity check applies to the actual input.
    Returns -1.0 if the score cannot be computed.
    """
    mask = labels != -1
    X_clean = X[mask]
    labels_clean = labels[mask]

    # Need at least 2 distinct labels and a minimum number of points
    if len(np.unique(labels_clean)) < 2 or len(labels_clean) < 4:
        return -1.0

    # Manual subsample — we control what goes into silhouette_score
    n = len(labels_clean)
    if n > sample_size:
        rng = np.random.default_rng(random_state)
        idx = rng.choice(n, size=sample_size, replace=False)
        X_clean = X_clean[idx]
        labels_clean = labels_clean[idx]

    # Re-check after subsampling — bad luck can still drop a label
    if len(np.unique(labels_clean)) < 2:
        return -1.0

    # No sample_size arg: we already did the sampling above
    return silhouette_score(X_clean, labels_clean)



In [ ]:

# ---- Cell 2: PCA for Dimensionality Reduction ----

print("=" * 60)
print("PRINCIPAL COMPONENT ANALYSIS (PCA)")
print("=" * 60)

pca_full = PCA()
pca_full.fit(X_scaled)

cumulative_variance = np.cumsum(pca_full.explained_variance_ratio_)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(range(1, len(pca_full.explained_variance_ratio_) + 1),
            pca_full.explained_variance_ratio_, color='steelblue', alpha=0.7)
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Variance Explained Ratio')
axes[0].set_title('Variance Explained by Each Component')

axes[1].plot(range(1, len(cumulative_variance) + 1), cumulative_variance,
             'bo-', markersize=4)
axes[1].axhline(y=0.95, color='r', linestyle='--', label='95% threshold')
axes[1].axhline(y=0.90, color='orange', linestyle='--', label='90% threshold')
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Cumulative Variance Explained')
axes[1].set_title('Cumulative Variance Explained')
axes[1].legend()

plt.tight_layout()
plt.savefig('../data/pca_variance.png', dpi=150, bbox_inches='tight')
plt.show()

n_components_95 = np.argmax(cumulative_variance >= 0.95) + 1
n_components_90 = np.argmax(cumulative_variance >= 0.90) + 1

print(f"\nComponents for 90% variance: {n_components_90}")
print(f"Components for 95% variance: {n_components_95}")

print("\nVariance explained by first 10 components:")
for i in range(min(10, len(pca_full.explained_variance_ratio_))):
    print(f"  PC{i+1}: {pca_full.explained_variance_ratio_[i]:.4f} "
          f"(cumulative: {cumulative_variance[i]:.4f})")


In [ ]:

# ---- Cell 3: Apply PCA with Selected Components ----

n_components = n_components_95
print(f"\nApplying PCA with {n_components} components")

pca = PCA(n_components=n_components)
X_pca = pca.fit_transform(X_scaled)

print(f"Original feature space: {X_scaled.shape[1]} dimensions")
print(f"Reduced feature space: {X_pca.shape[1]} dimensions")
print(f"Variance retained: {pca.explained_variance_ratio_.sum():.4f}")

pca_2d = PCA(n_components=2)
X_pca_2d = pca_2d.fit_transform(X_scaled)
print(f"\n2D PCA variance explained: {pca_2d.explained_variance_ratio_.sum():.4f}")

pca_3d = PCA(n_components=3)
X_pca_3d = pca_3d.fit_transform(X_scaled)
print(f"3D PCA variance explained: {pca_3d.explained_variance_ratio_.sum():.4f}")


In [ ]:

# ---- Cell 4: PCA Component Analysis ----

print("\n" + "=" * 60)
print("PCA COMPONENT LOADINGS")
print("=" * 60)

for pc_idx in range(3):
    loadings = pca_full.components_[pc_idx]
    sorted_idx = np.argsort(np.abs(loadings))[::-1]

    print(f"\nPC{pc_idx+1} - Top contributing features:")
    for i in range(5):
        feat_idx = sorted_idx[i]
        print(f"  {feature_names[feat_idx]:30s}: {loadings[feat_idx]:+.4f}")

fig, ax = plt.subplots(figsize=(16, 6))
loadings_df = pd.DataFrame(
    pca_full.components_[:5],
    columns=feature_names,
    index=[f'PC{i+1}' for i in range(5)]
)
sns.heatmap(loadings_df, cmap='RdBu_r', center=0, annot=True,
            fmt='.2f', ax=ax, xticklabels=True)
ax.set_title('PCA Component Loadings (First 5 Components)')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.tight_layout()
plt.savefig('../data/pca_loadings.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:


# ---- Cell 5: Determine DBSCAN Epsilon Parameter ----

print("=" * 60)
print("DETERMINING DBSCAN PARAMETERS")
print("=" * 60)

k = 2 * n_components
print(f"Using k = {k} for k-NN distance plot")

nn = NearestNeighbors(n_neighbors=k)
nn.fit(X_pca)
distances, indices = nn.kneighbors(X_pca)

k_distances = np.sort(distances[:, -1])

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(range(len(k_distances)), k_distances, color='steelblue', linewidth=0.5)
ax.set_xlabel('Points (sorted by distance)')
ax.set_ylabel(f'{k}-th Nearest Neighbor Distance')
ax.set_title(f'K-Distance Plot (k={k}) for Epsilon Selection')

for eps_candidate in [1.5, 2.0, 2.5, 3.0]:
    ax.axhline(y=eps_candidate, color='red', linestyle='--', alpha=0.5,
               label=f'ε = {eps_candidate}')

ax.legend()
plt.tight_layout()
plt.savefig('../data/k_distance_plot.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nK-distance statistics:")
print(f"  Mean: {k_distances.mean():.3f}")
print(f"  Median: {np.median(k_distances):.3f}")
print(f"  90th percentile: {np.percentile(k_distances, 90):.3f}")
print(f"  95th percentile: {np.percentile(k_distances, 95):.3f}")


In [ ]:

# ---- Cell 6: DBSCAN Parameter Grid Search ----

print("\n" + "=" * 60)
print("DBSCAN PARAMETER SEARCH")
print("=" * 60)

eps_values = [1.5, 2.0, 2.5, 3.0, 3.5, 4.0]
min_samples_values = [5, 10, 15, 20, 30]

results = []

for eps in eps_values:
    for min_samp in min_samples_values:
        dbscan = DBSCAN(eps=eps, min_samples=min_samp, n_jobs=-1)
        labels = dbscan.fit_predict(X_pca)

        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        n_noise = (labels == -1).sum()
        noise_ratio = n_noise / len(labels)

        if n_clusters >= 2 and noise_ratio < 0.95:
            sil_score = safe_silhouette(X_pca, labels)
        else:
            sil_score = -1.0

        results.append({
            'eps': eps,
            'min_samples': min_samp,
            'n_clusters': n_clusters,
            'n_noise': n_noise,
            'noise_ratio': noise_ratio,
            'silhouette': sil_score
        })

        print(f"eps={eps:.1f}, min_samples={min_samp:3d} -> "
              f"Clusters: {n_clusters:3d}, Noise: {noise_ratio:.3f}, "
              f"Silhouette: {sil_score:.4f}")

results_df = pd.DataFrame(results)

valid_results = results_df[
    (results_df['n_clusters'] >= 3) &
    (results_df['n_clusters'] <= 20) &
    (results_df['noise_ratio'] < 0.3) &
    (results_df['silhouette'] > 0)
]

if len(valid_results) > 0:
    best_idx = valid_results['silhouette'].idxmax()
    best_params = valid_results.loc[best_idx]
    print(f"\nBest DBSCAN parameters:")
    print(f"  eps = {best_params['eps']}")
    print(f"  min_samples = {int(best_params['min_samples'])}")
    print(f"  Clusters = {int(best_params['n_clusters'])}")
    print(f"  Noise ratio = {best_params['noise_ratio']:.3f}")
    print(f"  Silhouette = {best_params['silhouette']:.4f}")

    best_eps = best_params['eps']
    best_min_samples = int(best_params['min_samples'])
else:
    print("\nNo valid results found. Using default parameters.")
    best_eps = 2.5
    best_min_samples = 15

In [ ]:


# ---- Cell 7: Run Final DBSCAN ----

print("\n" + "=" * 60)
print("FINAL DBSCAN CLUSTERING")
print("=" * 60)

dbscan_final = DBSCAN(eps=best_eps, min_samples=best_min_samples, n_jobs=-1)
dbscan_labels = dbscan_final.fit_predict(X_pca)

n_clusters_dbscan = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
n_noise_dbscan = (dbscan_labels == -1).sum()

print(f"DBSCAN Results:")
print(f"  Parameters: eps={best_eps}, min_samples={best_min_samples}")
print(f"  Number of clusters: {n_clusters_dbscan}")
print(f"  Noise points: {n_noise_dbscan} ({n_noise_dbscan/len(dbscan_labels)*100:.2f}%)")

print(f"\nCluster sizes:")
for label in sorted(set(dbscan_labels)):
    count = (dbscan_labels == label).sum()
    name = "Noise" if label == -1 else f"Cluster {label}"
    print(f"  {name}: {count} points ({count/len(dbscan_labels)*100:.2f}%)")


In [ ]:

# ---- Cell 8: HDBSCAN Clustering ----

print("\n" + "=" * 60)
print("HDBSCAN CLUSTERING")
print("=" * 60)

hdbscan_results = []

for min_cluster_size in [50, 100, 200, 300, 500]:
    for min_samples in [5, 10, 15, 25]:
        clusterer = hdbscan.HDBSCAN(
            min_cluster_size=min_cluster_size,
            min_samples=min_samples,
            cluster_selection_method='eom',
            metric='euclidean'
        )
        hdb_labels = clusterer.fit_predict(X_pca)

        n_clusters = len(set(hdb_labels)) - (1 if -1 in hdb_labels else 0)
        n_noise = (hdb_labels == -1).sum()
        noise_ratio = n_noise / len(hdb_labels)

        if n_clusters >= 2 and noise_ratio < 0.95:
            sil = safe_silhouette(X_pca, hdb_labels)
        else:
            sil = -1.0

        hdbscan_results.append({
            'min_cluster_size': min_cluster_size,
            'min_samples': min_samples,
            'n_clusters': n_clusters,
            'noise_ratio': noise_ratio,
            'silhouette': sil
        })

        print(f"min_cluster_size={min_cluster_size:4d}, min_samples={min_samples:3d} -> "
              f"Clusters: {n_clusters:3d}, Noise: {noise_ratio:.3f}, Sil: {sil:.4f}")

hdbscan_results_df = pd.DataFrame(hdbscan_results)

valid_hdb = hdbscan_results_df[
    (hdbscan_results_df['n_clusters'] >= 3) &
    (hdbscan_results_df['n_clusters'] <= 15) &
    (hdbscan_results_df['noise_ratio'] < 0.3) &
    (hdbscan_results_df['silhouette'] > 0)
]

if len(valid_hdb) > 0:
    best_hdb_idx = valid_hdb['silhouette'].idxmax()
    best_hdb_params = valid_hdb.loc[best_hdb_idx]
    best_min_cluster = int(best_hdb_params['min_cluster_size'])
    best_hdb_min_samples = int(best_hdb_params['min_samples'])
else:
    best_min_cluster = 200
    best_hdb_min_samples = 10

print(f"\nBest HDBSCAN parameters:")
print(f"  min_cluster_size = {best_min_cluster}")
print(f"  min_samples = {best_hdb_min_samples}")


In [ ]:

# ---- Cell 9: Run Final HDBSCAN ----

print("\n" + "=" * 60)
print("FINAL HDBSCAN CLUSTERING")
print("=" * 60)

hdbscan_final = hdbscan.HDBSCAN(
    min_cluster_size=best_min_cluster,
    min_samples=best_hdb_min_samples,
    cluster_selection_method='eom',
    metric='euclidean',
    prediction_data=True
)
hdbscan_labels = hdbscan_final.fit_predict(X_pca)

n_clusters_hdbscan = len(set(hdbscan_labels)) - (1 if -1 in hdbscan_labels else 0)
n_noise_hdbscan = (hdbscan_labels == -1).sum()

print(f"HDBSCAN Results:")
print(f"  Number of clusters: {n_clusters_hdbscan}")
print(f"  Noise points: {n_noise_hdbscan} ({n_noise_hdbscan/len(hdbscan_labels)*100:.2f}%)")

print(f"\nCluster sizes:")
for label in sorted(set(hdbscan_labels)):
    count = (hdbscan_labels == label).sum()
    name = "Noise" if label == -1 else f"Cluster {label}"
    print(f"  {name}: {count} points ({count/len(hdbscan_labels)*100:.2f}%)")

if hasattr(hdbscan_final, 'cluster_persistence_'):
    print(f"\nCluster persistence (stability):")
    for i, persistence in enumerate(hdbscan_final.cluster_persistence_):
        print(f"  Cluster {i}: {persistence:.4f}")


In [ ]:

# ---- Cell 10: Compare DBSCAN vs HDBSCAN ----

print("\n" + "=" * 60)
print("CLUSTERING COMPARISON: DBSCAN vs HDBSCAN")
print("=" * 60)

comparison = {}

for name, labels in [("DBSCAN", dbscan_labels), ("HDBSCAN", hdbscan_labels)]:
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = (labels == -1).sum()
    mask = labels != -1

    metrics = {'n_clusters': n_clusters, 'noise_ratio': n_noise / len(labels)}

    # Check unique labels in the masked subset before computing any metric
    unique_masked = np.unique(labels[mask])
    if len(unique_masked) >= 2 and mask.sum() > 100:
        metrics['silhouette'] = safe_silhouette(X_pca, labels)
        metrics['calinski_harabasz'] = calinski_harabasz_score(X_pca[mask], labels[mask])
        metrics['davies_bouldin'] = davies_bouldin_score(X_pca[mask], labels[mask])

    comparison[name] = metrics
    print(f"\n{name}:")
    for k, v in metrics.items():
        print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

print("\n" + "-" * 40)
if ('silhouette' in comparison.get('DBSCAN', {}) and
        'silhouette' in comparison.get('HDBSCAN', {})):
    if comparison['DBSCAN']['silhouette'] > comparison['HDBSCAN']['silhouette']:
        best_algorithm = 'DBSCAN'
    else:
        best_algorithm = 'HDBSCAN'
    print(f"Better algorithm (by Silhouette Score): {best_algorithm}")
else:
    best_algorithm = 'HDBSCAN'
    print(f"Selected algorithm: {best_algorithm} (default)")

if best_algorithm == 'DBSCAN':
    final_labels = dbscan_labels
else:
    final_labels = hdbscan_labels

In [ ]:


# ---- Cell 11: Visualize Clusters in 2D PCA Space ----

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

scatter1 = axes[0].scatter(X_pca_2d[:, 0], X_pca_2d[:, 1],
                           c=dbscan_labels, cmap='Set1', s=1, alpha=0.3)
axes[0].set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]:.2%} var)')
axes[0].set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]:.2%} var)')
axes[0].set_title(f'DBSCAN Clustering ({n_clusters_dbscan} clusters)')
plt.colorbar(scatter1, ax=axes[0], label='Cluster')

scatter2 = axes[1].scatter(X_pca_2d[:, 0], X_pca_2d[:, 1],
                           c=hdbscan_labels, cmap='Set1', s=1, alpha=0.3)
axes[1].set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]:.2%} var)')
axes[1].set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]:.2%} var)')
axes[1].set_title(f'HDBSCAN Clustering ({n_clusters_hdbscan} clusters)')
plt.colorbar(scatter2, ax=axes[1], label='Cluster')

plt.tight_layout()
plt.savefig('../data/clustering_comparison_2d.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:

# ---- Cell 12: Cluster Profiling ----

print("\n" + "=" * 60)
print("CLUSTER PROFILING")
print("=" * 60)

temporal_df['cluster_dbscan'] = dbscan_labels
temporal_df['cluster_hdbscan'] = hdbscan_labels
temporal_df['cluster_final'] = final_labels

for cluster_id in sorted(set(final_labels)):
    mask = final_labels == cluster_id
    cluster_name = "NOISE" if cluster_id == -1 else f"CLUSTER {cluster_id}"

    print(f"\n{'='*40}")
    print(f"{cluster_name} (n={mask.sum()})")
    print(f"{'='*40}")

    cluster_data = temporal_df[mask]

    print(f"  Mean hour: {cluster_data['hour'].mean():.1f}")
    print(f"  Weekday fraction: {1 - cluster_data['is_weekend'].mean():.3f}")
    print(f"  Rush hour fraction: {cluster_data['is_rush_hour'].mean():.3f}")
    print(f"  Mean speed: {cluster_data['mean_speed'].mean():.1f} mph")
    print(f"  Std speed (across sensors): {cluster_data['std_speed'].mean():.1f} mph")
    print(f"  Fraction congested: {cluster_data['frac_congested'].mean():.3f}")
    print(f"  Fraction free flow: {cluster_data['frac_free_flow'].mean():.3f}")

    if 'time_period' in cluster_data.columns:
        print(f"  Most common time period: {cluster_data['time_period'].mode().values[0]}")

    mean_spd = cluster_data['mean_speed'].mean()
    rush_frac = cluster_data['is_rush_hour'].mean()

    if cluster_id == -1:
        interpretation = "Anomalous/Unusual Traffic Patterns"
    elif mean_spd > 55 and cluster_data['frac_free_flow'].mean() > 0.6:
        interpretation = "Free Flow Traffic"
    elif mean_spd < 35 and rush_frac > 0.5:
        interpretation = "Rush Hour Congestion"
    elif mean_spd < 35 and cluster_data['is_weekend'].mean() > 0.5:
        interpretation = "Weekend Congestion"
    elif 35 <= mean_spd <= 55:
        interpretation = "Moderate Traffic Flow"
    elif mean_spd > 55:
        interpretation = "Night/Off-Peak Free Flow"
    else:
        interpretation = "Mixed Traffic Conditions"

    print(f"  INTERPRETATION: {interpretation}")


In [ ]:

# ---- Cell 13: Save Clustering Results ----

# Apply HDF5-compatibility fixes before saving:
# drop string columns and cast booleans to int8
temporal_df_to_save = temporal_df.copy()

if 'time_period' in temporal_df_to_save.columns:
    temporal_df_to_save = temporal_df_to_save.drop(columns=['time_period'])

bool_cols = temporal_df_to_save.select_dtypes(include=['bool', 'boolean']).columns
for col in bool_cols:
    temporal_df_to_save[col] = temporal_df_to_save[col].astype('int8')

obj_cols = temporal_df_to_save.select_dtypes(include=['object']).columns
for col in obj_cols:
    temporal_df_to_save[col] = temporal_df_to_save[col].astype(str)

temporal_df_to_save.to_hdf('../data/temporal_features_clustered.h5', key='df', mode='w')
print(f"Clustered temporal features saved: {temporal_df_to_save.shape}")

np.save('../data/X_pca.npy', X_pca)
np.save('../data/X_pca_2d.npy', X_pca_2d)
np.save('../data/X_pca_3d.npy', X_pca_3d)
print("PCA-transformed data saved")

np.save('../data/dbscan_labels.npy', dbscan_labels)
np.save('../data/hdbscan_labels.npy', hdbscan_labels)
np.save('../data/final_labels.npy', final_labels)
print("Cluster labels saved")

with open('../data/pca_model.pkl', 'wb') as f:
    pickle.dump(pca, f)

with open('../data/clustering_comparison.json', 'w') as f:
    json.dump(comparison, f, indent=2, default=str)

print(f"\nBest algorithm: {best_algorithm}")
print(f"Number of clusters: {len(set(final_labels)) - (1 if -1 in final_labels else 0)}")

print("\n" + "=" * 60)
print("NOTEBOOK 3 COMPLETE - Proceed to Notebook 4")
print("=" * 60)

In [ ]:
# # =============================================================================
# # NOTEBOOK 3: UNSUPERVISED MODELLING
# # =============================================================================
# # Purpose: Apply DBSCAN and HDBSCAN clustering algorithms to discover
# # traffic pattern clusters. Use PCA for dimensionality reduction.
# # Evaluate clustering quality using internal metrics.
# # =============================================================================

# # ---- Cell 1: Import Libraries and Load Data ----

# import pandas as pd
# import numpy as np
# import matplotlib.pyplot as plt
# import seaborn as sns
# from sklearn.decomposition import PCA
# from sklearn.cluster import DBSCAN
# from sklearn.neighbors import NearestNeighbors
# from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
# import hdbscan
# import json
# import pickle
# import warnings
# warnings.filterwarnings('ignore')

# plt.style.use('seaborn-v0_8-whitegrid')

# # Load engineered features
# X_scaled = pd.read_hdf('../data/features_scaled.h5')
# temporal_df = pd.read_hdf('../data/temporal_features.h5')

# with open('../data/feature_names.json', 'r') as f:
#     feature_names = json.load(f)

# print(f"Feature matrix loaded: {X_scaled.shape}")
# print(f"Temporal features loaded: {temporal_df.shape}")
# print(f"Number of features: {len(feature_names)}")


# # ---- Shared Helper: Safe Silhouette Score ----
# # Passing sample_size to sklearn's silhouette_score lets it draw its own
# # internal subsample AFTER our guard has run — that subsample can still
# # collapse to a single cluster by bad luck and re-raise the ValueError.
# # Fix: subsample manually so we can re-check label diversity on the exact
# # array being passed, then call silhouette_score with no sample_size arg.

# def safe_silhouette(X, labels, sample_size=5000, random_state=42):
#     """
#     Compute silhouette score ignoring noise points (label == -1).
#     Subsamples manually so the diversity check applies to the actual input.
#     Returns -1.0 if the score cannot be computed.
#     """
#     mask = labels != -1
#     X_clean = X[mask]
#     labels_clean = labels[mask]

#     # Need at least 2 distinct labels and a minimum number of points
#     if len(np.unique(labels_clean)) < 2 or len(labels_clean) < 4:
#         return -1.0

#     # Manual subsample — we control what goes into silhouette_score
#     n = len(labels_clean)
#     if n > sample_size:
#         rng = np.random.default_rng(random_state)
#         idx = rng.choice(n, size=sample_size, replace=False)
#         X_clean = X_clean[idx]
#         labels_clean = labels_clean[idx]

#     # Re-check after subsampling — bad luck can still drop a label
#     if len(np.unique(labels_clean)) < 2:
#         return -1.0

#     # No sample_size arg: we already did the sampling above
#     return silhouette_score(X_clean, labels_clean)


# # ---- Cell 2: PCA for Dimensionality Reduction ----

# print("=" * 60)
# print("PRINCIPAL COMPONENT ANALYSIS (PCA)")
# print("=" * 60)

# pca_full = PCA()
# pca_full.fit(X_scaled)

# cumulative_variance = np.cumsum(pca_full.explained_variance_ratio_)

# fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# axes[0].bar(range(1, len(pca_full.explained_variance_ratio_) + 1),
#             pca_full.explained_variance_ratio_, color='steelblue', alpha=0.7)
# axes[0].set_xlabel('Principal Component')
# axes[0].set_ylabel('Variance Explained Ratio')
# axes[0].set_title('Variance Explained by Each Component')

# axes[1].plot(range(1, len(cumulative_variance) + 1), cumulative_variance,
#              'bo-', markersize=4)
# axes[1].axhline(y=0.95, color='r', linestyle='--', label='95% threshold')
# axes[1].axhline(y=0.90, color='orange', linestyle='--', label='90% threshold')
# axes[1].set_xlabel('Number of Components')
# axes[1].set_ylabel('Cumulative Variance Explained')
# axes[1].set_title('Cumulative Variance Explained')
# axes[1].legend()

# plt.tight_layout()
# plt.savefig('../data/pca_variance.png', dpi=150, bbox_inches='tight')
# plt.show()

# n_components_95 = np.argmax(cumulative_variance >= 0.95) + 1
# n_components_90 = np.argmax(cumulative_variance >= 0.90) + 1

# print(f"\nComponents for 90% variance: {n_components_90}")
# print(f"Components for 95% variance: {n_components_95}")

# print("\nVariance explained by first 10 components:")
# for i in range(min(10, len(pca_full.explained_variance_ratio_))):
#     print(f"  PC{i+1}: {pca_full.explained_variance_ratio_[i]:.4f} "
#           f"(cumulative: {cumulative_variance[i]:.4f})")

# # ---- Cell 3: Apply PCA with Selected Components ----

# n_components = n_components_95
# print(f"\nApplying PCA with {n_components} components")

# pca = PCA(n_components=n_components)
# X_pca = pca.fit_transform(X_scaled)

# print(f"Original feature space: {X_scaled.shape[1]} dimensions")
# print(f"Reduced feature space: {X_pca.shape[1]} dimensions")
# print(f"Variance retained: {pca.explained_variance_ratio_.sum():.4f}")

# pca_2d = PCA(n_components=2)
# X_pca_2d = pca_2d.fit_transform(X_scaled)
# print(f"\n2D PCA variance explained: {pca_2d.explained_variance_ratio_.sum():.4f}")

# pca_3d = PCA(n_components=3)
# X_pca_3d = pca_3d.fit_transform(X_scaled)
# print(f"3D PCA variance explained: {pca_3d.explained_variance_ratio_.sum():.4f}")

# # ---- Cell 4: PCA Component Analysis ----

# print("\n" + "=" * 60)
# print("PCA COMPONENT LOADINGS")
# print("=" * 60)

# for pc_idx in range(3):
#     loadings = pca_full.components_[pc_idx]
#     sorted_idx = np.argsort(np.abs(loadings))[::-1]

#     print(f"\nPC{pc_idx+1} - Top contributing features:")
#     for i in range(5):
#         feat_idx = sorted_idx[i]
#         print(f"  {feature_names[feat_idx]:30s}: {loadings[feat_idx]:+.4f}")

# fig, ax = plt.subplots(figsize=(16, 6))
# loadings_df = pd.DataFrame(
#     pca_full.components_[:5],
#     columns=feature_names,
#     index=[f'PC{i+1}' for i in range(5)]
# )
# sns.heatmap(loadings_df, cmap='RdBu_r', center=0, annot=True,
#             fmt='.2f', ax=ax, xticklabels=True)
# ax.set_title('PCA Component Loadings (First 5 Components)')
# plt.xticks(rotation=45, ha='right', fontsize=8)
# plt.tight_layout()
# plt.savefig('../data/pca_loadings.png', dpi=150, bbox_inches='tight')
# plt.show()

# # ---- Cell 5: Determine DBSCAN Epsilon Parameter ----

# print("=" * 60)
# print("DETERMINING DBSCAN PARAMETERS")
# print("=" * 60)

# k = 2 * n_components
# print(f"Using k = {k} for k-NN distance plot")

# nn = NearestNeighbors(n_neighbors=k)
# nn.fit(X_pca)
# distances, indices = nn.kneighbors(X_pca)

# k_distances = np.sort(distances[:, -1])

# fig, ax = plt.subplots(figsize=(12, 6))
# ax.plot(range(len(k_distances)), k_distances, color='steelblue', linewidth=0.5)
# ax.set_xlabel('Points (sorted by distance)')
# ax.set_ylabel(f'{k}-th Nearest Neighbor Distance')
# ax.set_title(f'K-Distance Plot (k={k}) for Epsilon Selection')

# for eps_candidate in [1.5, 2.0, 2.5, 3.0]:
#     ax.axhline(y=eps_candidate, color='red', linestyle='--', alpha=0.5,
#                label=f'ε = {eps_candidate}')

# ax.legend()
# plt.tight_layout()
# plt.savefig('../data/k_distance_plot.png', dpi=150, bbox_inches='tight')
# plt.show()

# print(f"\nK-distance statistics:")
# print(f"  Mean: {k_distances.mean():.3f}")
# print(f"  Median: {np.median(k_distances):.3f}")
# print(f"  90th percentile: {np.percentile(k_distances, 90):.3f}")
# print(f"  95th percentile: {np.percentile(k_distances, 95):.3f}")

# # ---- Cell 6: DBSCAN Parameter Grid Search ----

# print("\n" + "=" * 60)
# print("DBSCAN PARAMETER SEARCH")
# print("=" * 60)

# eps_values = [1.5, 2.0, 2.5, 3.0, 3.5, 4.0]
# min_samples_values = [5, 10, 15, 20, 30]

# results = []

# for eps in eps_values:
#     for min_samp in min_samples_values:
#         dbscan = DBSCAN(eps=eps, min_samples=min_samp, n_jobs=-1)
#         labels = dbscan.fit_predict(X_pca)

#         n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
#         n_noise = (labels == -1).sum()
#         noise_ratio = n_noise / len(labels)

#         if n_clusters >= 2 and noise_ratio < 0.95:
#             sil_score = safe_silhouette(X_pca, labels)
#         else:
#             sil_score = -1.0

#         results.append({
#             'eps': eps,
#             'min_samples': min_samp,
#             'n_clusters': n_clusters,
#             'n_noise': n_noise,
#             'noise_ratio': noise_ratio,
#             'silhouette': sil_score
#         })

#         print(f"eps={eps:.1f}, min_samples={min_samp:3d} -> "
#               f"Clusters: {n_clusters:3d}, Noise: {noise_ratio:.3f}, "
#               f"Silhouette: {sil_score:.4f}")

# results_df = pd.DataFrame(results)

# valid_results = results_df[
#     (results_df['n_clusters'] >= 3) &
#     (results_df['n_clusters'] <= 20) &
#     (results_df['noise_ratio'] < 0.3) &
#     (results_df['silhouette'] > 0)
# ]

# if len(valid_results) > 0:
#     best_idx = valid_results['silhouette'].idxmax()
#     best_params = valid_results.loc[best_idx]
#     print(f"\nBest DBSCAN parameters:")
#     print(f"  eps = {best_params['eps']}")
#     print(f"  min_samples = {int(best_params['min_samples'])}")
#     print(f"  Clusters = {int(best_params['n_clusters'])}")
#     print(f"  Noise ratio = {best_params['noise_ratio']:.3f}")
#     print(f"  Silhouette = {best_params['silhouette']:.4f}")

#     best_eps = best_params['eps']
#     best_min_samples = int(best_params['min_samples'])
# else:
#     print("\nNo valid results found. Using default parameters.")
#     best_eps = 2.5
#     best_min_samples = 15

# # ---- Cell 7: Run Final DBSCAN ----

# print("\n" + "=" * 60)
# print("FINAL DBSCAN CLUSTERING")
# print("=" * 60)

# dbscan_final = DBSCAN(eps=best_eps, min_samples=best_min_samples, n_jobs=-1)
# dbscan_labels = dbscan_final.fit_predict(X_pca)

# n_clusters_dbscan = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
# n_noise_dbscan = (dbscan_labels == -1).sum()

# print(f"DBSCAN Results:")
# print(f"  Parameters: eps={best_eps}, min_samples={best_min_samples}")
# print(f"  Number of clusters: {n_clusters_dbscan}")
# print(f"  Noise points: {n_noise_dbscan} ({n_noise_dbscan/len(dbscan_labels)*100:.2f}%)")

# print(f"\nCluster sizes:")
# for label in sorted(set(dbscan_labels)):
#     count = (dbscan_labels == label).sum()
#     name = "Noise" if label == -1 else f"Cluster {label}"
#     print(f"  {name}: {count} points ({count/len(dbscan_labels)*100:.2f}%)")

# # ---- Cell 8: HDBSCAN Clustering ----

# print("\n" + "=" * 60)
# print("HDBSCAN CLUSTERING")
# print("=" * 60)

# hdbscan_results = []

# for min_cluster_size in [50, 100, 200, 300, 500]:
#     for min_samples in [5, 10, 15, 25]:
#         clusterer = hdbscan.HDBSCAN(
#             min_cluster_size=min_cluster_size,
#             min_samples=min_samples,
#             cluster_selection_method='eom',
#             metric='euclidean'
#         )
#         hdb_labels = clusterer.fit_predict(X_pca)

#         n_clusters = len(set(hdb_labels)) - (1 if -1 in hdb_labels else 0)
#         n_noise = (hdb_labels == -1).sum()
#         noise_ratio = n_noise / len(hdb_labels)

#         if n_clusters >= 2 and noise_ratio < 0.95:
#             sil = safe_silhouette(X_pca, hdb_labels)
#         else:
#             sil = -1.0

#         hdbscan_results.append({
#             'min_cluster_size': min_cluster_size,
#             'min_samples': min_samples,
#             'n_clusters': n_clusters,
#             'noise_ratio': noise_ratio,
#             'silhouette': sil
#         })

#         print(f"min_cluster_size={min_cluster_size:4d}, min_samples={min_samples:3d} -> "
#               f"Clusters: {n_clusters:3d}, Noise: {noise_ratio:.3f}, Sil: {sil:.4f}")

# hdbscan_results_df = pd.DataFrame(hdbscan_results)

# valid_hdb = hdbscan_results_df[
#     (hdbscan_results_df['n_clusters'] >= 3) &
#     (hdbscan_results_df['n_clusters'] <= 15) &
#     (hdbscan_results_df['noise_ratio'] < 0.3) &
#     (hdbscan_results_df['silhouette'] > 0)
# ]

# if len(valid_hdb) > 0:
#     best_hdb_idx = valid_hdb['silhouette'].idxmax()
#     best_hdb_params = valid_hdb.loc[best_hdb_idx]
#     best_min_cluster = int(best_hdb_params['min_cluster_size'])
#     best_hdb_min_samples = int(best_hdb_params['min_samples'])
# else:
#     best_min_cluster = 200
#     best_hdb_min_samples = 10

# print(f"\nBest HDBSCAN parameters:")
# print(f"  min_cluster_size = {best_min_cluster}")
# print(f"  min_samples = {best_hdb_min_samples}")

# # ---- Cell 9: Run Final HDBSCAN ----

# print("\n" + "=" * 60)
# print("FINAL HDBSCAN CLUSTERING")
# print("=" * 60)

# hdbscan_final = hdbscan.HDBSCAN(
#     min_cluster_size=best_min_cluster,
#     min_samples=best_hdb_min_samples,
#     cluster_selection_method='eom',
#     metric='euclidean',
#     prediction_data=True
# )
# hdbscan_labels = hdbscan_final.fit_predict(X_pca)

# n_clusters_hdbscan = len(set(hdbscan_labels)) - (1 if -1 in hdbscan_labels else 0)
# n_noise_hdbscan = (hdbscan_labels == -1).sum()

# print(f"HDBSCAN Results:")
# print(f"  Number of clusters: {n_clusters_hdbscan}")
# print(f"  Noise points: {n_noise_hdbscan} ({n_noise_hdbscan/len(hdbscan_labels)*100:.2f}%)")

# print(f"\nCluster sizes:")
# for label in sorted(set(hdbscan_labels)):
#     count = (hdbscan_labels == label).sum()
#     name = "Noise" if label == -1 else f"Cluster {label}"
#     print(f"  {name}: {count} points ({count/len(hdbscan_labels)*100:.2f}%)")

# if hasattr(hdbscan_final, 'cluster_persistence_'):
#     print(f"\nCluster persistence (stability):")
#     for i, persistence in enumerate(hdbscan_final.cluster_persistence_):
#         print(f"  Cluster {i}: {persistence:.4f}")

# # ---- Cell 10: Compare DBSCAN vs HDBSCAN ----

# print("\n" + "=" * 60)
# print("CLUSTERING COMPARISON: DBSCAN vs HDBSCAN")
# print("=" * 60)

# comparison = {}

# for name, labels in [("DBSCAN", dbscan_labels), ("HDBSCAN", hdbscan_labels)]:
#     n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
#     n_noise = (labels == -1).sum()
#     mask = labels != -1

#     metrics = {'n_clusters': n_clusters, 'noise_ratio': n_noise / len(labels)}

#     # Check unique labels in the masked subset before computing any metric
#     unique_masked = np.unique(labels[mask])
#     if len(unique_masked) >= 2 and mask.sum() > 100:
#         metrics['silhouette'] = safe_silhouette(X_pca, labels)
#         metrics['calinski_harabasz'] = calinski_harabasz_score(X_pca[mask], labels[mask])
#         metrics['davies_bouldin'] = davies_bouldin_score(X_pca[mask], labels[mask])

#     comparison[name] = metrics
#     print(f"\n{name}:")
#     for k, v in metrics.items():
#         print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

# print("\n" + "-" * 40)
# if ('silhouette' in comparison.get('DBSCAN', {}) and
#         'silhouette' in comparison.get('HDBSCAN', {})):
#     if comparison['DBSCAN']['silhouette'] > comparison['HDBSCAN']['silhouette']:
#         best_algorithm = 'DBSCAN'
#     else:
#         best_algorithm = 'HDBSCAN'
#     print(f"Better algorithm (by Silhouette Score): {best_algorithm}")
# else:
#     best_algorithm = 'HDBSCAN'
#     print(f"Selected algorithm: {best_algorithm} (default)")

# if best_algorithm == 'DBSCAN':
#     final_labels = dbscan_labels
# else:
#     final_labels = hdbscan_labels

# # ---- Cell 11: Visualize Clusters in 2D PCA Space ----

# fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# scatter1 = axes[0].scatter(X_pca_2d[:, 0], X_pca_2d[:, 1],
#                            c=dbscan_labels, cmap='Set1', s=1, alpha=0.3)
# axes[0].set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]:.2%} var)')
# axes[0].set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]:.2%} var)')
# axes[0].set_title(f'DBSCAN Clustering ({n_clusters_dbscan} clusters)')
# plt.colorbar(scatter1, ax=axes[0], label='Cluster')

# scatter2 = axes[1].scatter(X_pca_2d[:, 0], X_pca_2d[:, 1],
#                            c=hdbscan_labels, cmap='Set1', s=1, alpha=0.3)
# axes[1].set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]:.2%} var)')
# axes[1].set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]:.2%} var)')
# axes[1].set_title(f'HDBSCAN Clustering ({n_clusters_hdbscan} clusters)')
# plt.colorbar(scatter2, ax=axes[1], label='Cluster')

# plt.tight_layout()
# plt.savefig('../data/clustering_comparison_2d.png', dpi=150, bbox_inches='tight')
# plt.show()

# # ---- Cell 12: Cluster Profiling ----

# print("\n" + "=" * 60)
# print("CLUSTER PROFILING")
# print("=" * 60)

# temporal_df['cluster_dbscan'] = dbscan_labels
# temporal_df['cluster_hdbscan'] = hdbscan_labels
# temporal_df['cluster_final'] = final_labels

# for cluster_id in sorted(set(final_labels)):
#     mask = final_labels == cluster_id
#     cluster_name = "NOISE" if cluster_id == -1 else f"CLUSTER {cluster_id}"

#     print(f"\n{'='*40}")
#     print(f"{cluster_name} (n={mask.sum()})")
#     print(f"{'='*40}")

#     cluster_data = temporal_df[mask]

#     print(f"  Mean hour: {cluster_data['hour'].mean():.1f}")
#     print(f"  Weekday fraction: {1 - cluster_data['is_weekend'].mean():.3f}")
#     print(f"  Rush hour fraction: {cluster_data['is_rush_hour'].mean():.3f}")
#     print(f"  Mean speed: {cluster_data['mean_speed'].mean():.1f} mph")
#     print(f"  Std speed (across sensors): {cluster_data['std_speed'].mean():.1f} mph")
#     print(f"  Fraction congested: {cluster_data['frac_congested'].mean():.3f}")
#     print(f"  Fraction free flow: {cluster_data['frac_free_flow'].mean():.3f}")

#     if 'time_period' in cluster_data.columns:
#         print(f"  Most common time period: {cluster_data['time_period'].mode().values[0]}")

#     mean_spd = cluster_data['mean_speed'].mean()
#     rush_frac = cluster_data['is_rush_hour'].mean()

#     if cluster_id == -1:
#         interpretation = "Anomalous/Unusual Traffic Patterns"
#     elif mean_spd > 55 and cluster_data['frac_free_flow'].mean() > 0.6:
#         interpretation = "Free Flow Traffic"
#     elif mean_spd < 35 and rush_frac > 0.5:
#         interpretation = "Rush Hour Congestion"
#     elif mean_spd < 35 and cluster_data['is_weekend'].mean() > 0.5:
#         interpretation = "Weekend Congestion"
#     elif 35 <= mean_spd <= 55:
#         interpretation = "Moderate Traffic Flow"
#     elif mean_spd > 55:
#         interpretation = "Night/Off-Peak Free Flow"
#     else:
#         interpretation = "Mixed Traffic Conditions"

#     print(f"  INTERPRETATION: {interpretation}")

# # ---- Cell 13: Save Clustering Results ----

# # Apply HDF5-compatibility fixes before saving:
# # drop string columns and cast booleans to int8
# temporal_df_to_save = temporal_df.copy()

# if 'time_period' in temporal_df_to_save.columns:
#     temporal_df_to_save = temporal_df_to_save.drop(columns=['time_period'])

# bool_cols = temporal_df_to_save.select_dtypes(include=['bool', 'boolean']).columns
# for col in bool_cols:
#     temporal_df_to_save[col] = temporal_df_to_save[col].astype('int8')

# obj_cols = temporal_df_to_save.select_dtypes(include=['object']).columns
# for col in obj_cols:
#     temporal_df_to_save[col] = temporal_df_to_save[col].astype(str)

# temporal_df_to_save.to_hdf('../data/temporal_features_clustered.h5', key='df', mode='w')
# print(f"Clustered temporal features saved: {temporal_df_to_save.shape}")

# np.save('../data/X_pca.npy', X_pca)
# np.save('../data/X_pca_2d.npy', X_pca_2d)
# np.save('../data/X_pca_3d.npy', X_pca_3d)
# print("PCA-transformed data saved")

# np.save('../data/dbscan_labels.npy', dbscan_labels)
# np.save('../data/hdbscan_labels.npy', hdbscan_labels)
# np.save('../data/final_labels.npy', final_labels)
# print("Cluster labels saved")

# with open('../data/pca_model.pkl', 'wb') as f:
#     pickle.dump(pca, f)

# with open('../data/clustering_comparison.json', 'w') as f:
#     json.dump(comparison, f, indent=2, default=str)

# print(f"\nBest algorithm: {best_algorithm}")
# print(f"Number of clusters: {len(set(final_labels)) - (1 if -1 in final_labels else 0)}")

# print("\n" + "=" * 60)
# print("NOTEBOOK 3 COMPLETE - Proceed to Notebook 4")
# print("=" * 60)